In [2]:

#DESeq and PCA and Dandogram 

library("DESeq2")
library("gplots")
library("RColorBrewer")
library(dplyr)
library(tidyr)
library(ggplot2)
library(ggrepel)

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, append, as.data.frame, basename, cbind, colnames,
    dirname, do.call, duplicated, eval, evalq, Filter, Find, get, grep,
    grepl, intersect, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, setdiff, sort, table, tapply,
    union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:base’:

    expand.grid, I, unname


Loading required package: IRanges

Loading required package: GenomicRanges

Loading required package: GenomeInfoDb

Loading required package: SummarizedExperiment

Loading required package: MatrixGe

In [4]:
#set path to analysis folder 
setwd("/lustre/scratch123/hgi/projects/mouse_epi/users/naru/inProgres/mice_PiRNA/workflow/scripts/R")

In [5]:
# Make a color scheme for heatmaps
hmcol = colorRampPalette(brewer.pal(9, "GnBu"))(100)


In [6]:
#Input matrix of raw counts
mircounts <- read.table("mirCounts.txt",header=TRUE,row.names=1)
mircounts

,one_hESC_1,two_hESC_2,three_hPGCLC_1,four_hPGCLC_2,five_TCAM2_1,six_TCAM2_2,seven_hPGC_Wk6_F_1,eight_hPGC_Wk6_F_2,nine_hPGC_F_1,ten_hPGC_F_2,⋯,hPGC_Wk11to13_F_2,hPGC_Wk11to13_M_1,hPGC_Wk11to13_M_2,hPGC_Wk14to17_F_1,hPGC_Wk14to17_F_2,hPGC_Wk14to17_M_1,hPGC_Wk18to22_F_1,hPGC_Wk18to22_F_2,hPGC_Wk18to22_M_1,hPGC_Wk18to22_M_2
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
let-7a-2-3p,0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
let-7a-3p,0,0,0,0,13,11,4,0,0,0,⋯,24,42,45,2,32,60,44,24,83,243
let-7a-5p,5454,2652,3474,3422,10556,6594,2831,1804,4353,2322,⋯,19021,33360,36979,7252,28680,44200,50457,38624,86501,226686
let-7b-3p,0,0,0,0,0,0,0,0,0,0,⋯,10,21,54,4,46,81,87,45,152,391
let-7b-5p,2013,1002,337,718,42,11,98,144,505,175,⋯,5280,5510,7959,2574,12812,14355,24397,20288,49394,144185
let-7c-3p,0,0,0,0,15,3,0,0,0,0,⋯,4,0,0,0,0,15,4,0,7,44
let-7c-5p,1181,90,133,108,199,100,983,633,961,492,⋯,4520,8730,8220,2054,14147,12948,19622,17043,28268,62282
let-7d-3p,46,7,15,0,45,37,16,14,0,54,⋯,250,425,471,77,406,800,652,497,1189,2651
let-7d-5p,783,14,69,118,144,124,83,49,202,153,⋯,2287,3740,4193,800,2934,5177,5897,3724,9533,18889


In [ ]:


samplenames <- c("4i_hESC", "4i_hESC", "hPGCLC", "hPGCLC", "TCAM2", "TCAM2", 
                "hPGC_F_Wk6", "hPGC_F_Wk6", "hPGC_F_Wk7", "hPGC_F_Wk7", "hPGC_F_Wk9", "hPGC_F_Wk9",
                "hPGC_M_Wk6", "hPGC_M_Wk6", "hPGC_M_Wk7", "hPGC_M_Wk7", "hPGC_M_Wk9", "hPGC_M_Wk9",
                "Soma_F_Wk6", "Soma_F_Wk6", "Soma_F_Wk7", "Soma_F_Wk7", "Soma_F_Wk9", "Soma_F_Wk9",
                "Soma_M_Wk6", "Soma_M_Wk6", "Soma_M_Wk7", "Soma_M_Wk7", "Soma_M_Wk9", "Soma_M_Wk9",
                "E8_hESC", "E8_hESC", "pre_ME", "pre_ME", "naive_hESC", "naive_hESC", 
                "hPGC_M_Wk7",
                "hPGC_F_Wk11to13", "hPGC_F_Wk11to13", "hPGC_M_Wk11to13", "hPGC_M_Wk11to13",
                "hPGC_F_Wk14to17", "hPGC_F_Wk14to17", "hPGC_M_Wk14to17",
                "hPGC_F_Wk18to22", "hPGC_F_Wk18to22", "hPGC_M_Wk18to22", "hPGC_M_Wk18to22",
                "Soma_M_Wk7")


colnames(mircounts) = paste(c("4i_hESC_1", "4i_hESC_2", "hPGCLC_1", "hPGCLC_2", "TCAM2_1", "TCAM2_2", 
                              "hPGC_F_Wk6_1", "hPGC_F_Wk6_2", "hPGC_F_Wk7_1", "hPGC_F_Wk7_2", "hPGC_F_Wk9_1", "hPGC_F_Wk9_2",
                              "hPGC_M_Wk6_1", "hPGC_M_Wk6_2", "hPGC_M_Wk7_1", "hPGC_M_Wk7_2", "hPGC_M_Wk9_1", "hPGC_M_Wk9_2",
                              "Soma_F_Wk6_1", "Soma_F_Wk6_2", "Soma_F_Wk7_1", "Soma_F_Wk7_2", "Soma_F_Wk9_1", "Soma_F_Wk9_2",
                              "Soma_M_Wk6_1", "Soma_M_Wk6_2", "Soma_M_Wk7_1", "Soma_M_Wk7_2", "Soma_M_Wk9_1", "Soma_M_Wk9_2",
                              "E8_hESC_1", "E8_hESC_2", "pre_ME_1", "pre_ME_2", "naive_hESC_1", "naive_hESC_2", 
                              "hPGC_M_Wk7_1_rerun",
                              "hPGC_F_Wk11to13_1", "hPGC_F_Wk11to13_2", "hPGC_M_Wk11to13_1", "hPGC_M_Wk11to13_2",
                              "hPGC_F_Wk14to17_1", "hPGC_F_Wk14to17_2", "hPGC_M_Wk14to17_1",
                              "hPGC_F_Wk18to22_1", "hPGC_F_Wk18to22_2", "hPGC_M_Wk18to22_1", "hPGC_M_Wk18to22_2",
                              "Soma_M_Wk7_2_rerun"))
head(mircounts)


In [ ]:
# First we tell DESeq which samples correspond to which tissues.
conds = data.frame(samplenames) ##what Desq2 manual calls coldata
colnames(conds)="sample"



In [ ]:

# Now we build a DESeq2 Count dataset and normalize it.
cds <- DESeqDataSetFromMatrix(countData = mircounts, colData = conds, design = ~ sample)
cds <- estimateSizeFactors(cds)
cds <- estimateDispersions(cds)
cds <- nbinomWaldTest(cds)


In [ ]:
pdf("BarGraphBeforeAfter.pdf")
par(mfrow=c(2,1))
prenorm=apply(mircounts,2,sum)
barplot(prenorm,col=as.factor(samplenames),las=2,names=samplenames)
postnorm=apply(counts(cds,normalized=TRUE),2,sum)
barplot(postnorm,col=as.factor(samplenames),las=2,names=samplenames)
dev.off()


In [ ]:
#rlog normalisations
rl=rlog(cds)


In [ ]:
######Making a subset of the data for PCA, and perhaps all samples.

rl.sub <- rl[ , rl$sample %in% c("Soma-F-Wk6", "Soma-F-Wk7", "Soma-F-Wk9",
                                 "Soma-M-Wk6", "Soma-M-Wk7", "Soma-M-Wk9") ]

rl.sub2 <- rl[ , rl$sample %in% c("hPGC-F-Wk6", "hPGC-F-Wk7", "hPGC-F-Wk9",
                                 "hPGC-M-Wk6", "hPGC-M-Wk7", "hPGC-M-Wk9") ]

rl.sub3 <- rl[ , rl$sample %in% c("Soma-F-Wk6", "Soma-F-Wk7", "Soma-F-Wk9",
                                  "Soma-M-Wk6", "Soma-M-Wk7", "Soma-M-Wk9", 
                                  "hPGC-F-Wk6", "hPGC-F-Wk7", "hPGC-F-Wk9",
                                  "hPGC-M-Wk6", "hPGC-M-Wk7", "hPGC-M-Wk9") ]
rl.sub4 <- rl[ , rl$sample %in% c("4i-hESC", "naive_hESC", "hPGCLC",
                                  "pre-ME", "hPGC-M-Wk7", "TCAM2", 
                                  "JEG3", "E8-hESC", "hPGC-F-Wk7") ]

rl.sub5 <- rl[ , rl$sample %in% c("4i-hESC", "naive_hESC", "hPGCLC",
                                  "pre-ME", "hPGC-M-Wk7", "TCAM2", 
                                  "E8-hESC", "hPGC-F-Wk7") ]


In [ ]:
##To plot PC1 versus 2 ## ALways change the colours to alphabetical order, and change the data.frame colData to rl.sub1 or approrpiate
plotPCA.san <- function (object, intgroup = "condition", ntop = 500, returnData = FALSE) 
{
  rv <- rowVars(assay(object))
  select <- order(rv, decreasing = TRUE)[seq_len(min(ntop, 
                                                     length(rv)))]
  pca <- prcomp(t(assay(object)[select, ]))
  percentVar <- pca$sdev^2/sum(pca$sdev^2)
  if (!all(intgroup %in% names(colData(object)))) {
    stop("the argument 'intgroup' should specify columns of colData(dds)")
  }
  intgroup.df <- as.data.frame(colData(object)[, intgroup, drop = FALSE])
  group <- if (length(intgroup) > 1) {
    factor(apply(intgroup.df, 1, paste, collapse = " : "))
  }
  else {
    colData(object)[[intgroup]]
  }
  d <- data.frame(PC1 = pca$x[, 1], PC2 = pca$x[, 2], group = group, 
                  intgroup.df, name = colData(rl.sub1)[,1])
  if (returnData) {
    attr(d, "percentVar") <- percentVar[1:2]
    return(d)
  }
  ggplot(data = d, aes_string(x = "PC1", y = "PC2", color = "group", label = "name")) + geom_point(size = 9) + xlab(paste0("PC1: ", round(percentVar[1] * 100), "% variance")) + 
    ylab(paste0("PC2: ", round(percentVar[2] * 100), "% variance")) + coord_fixed()+
    theme_bw()+
    scale_color_manual(values=c("indianred4", "thistle1", "plum3", "orchid3",
                                "darkseagreen2", "darkseagreen3", "aquamarine3",
                                "coral", 
                                "darkolivegreen3", "khaki3", "darkolivegreen4",
                                "tan", "tan2", "peru", 
                                "hotpink1"))
  
}

pdf("PCA_After_1vs2_Beginning.pdf")
plotPCA.san(rl.sub1, intgroup = "sample", ntop = nrow(counts(cds)))
dev.off()


In [ ]:

##To plot PC2 versus 3
library(genefilter)
library(ggplot2)
library(ggrepel)

plotPCA.san <- function (object, intgroup = "condition", ntop = 500, returnData = FALSE) 
{
  rv <- rowVars(assay(object))
  select <- order(rv, decreasing = TRUE)[seq_len(min(ntop, 
                                                     length(rv)))]
  pca <- prcomp(t(assay(object)[select, ]))
  percentVar <- pca$sdev^2/sum(pca$sdev^2)
  if (!all(intgroup %in% names(colData(object)))) {
    stop("the argument 'intgroup' should specify columns of colData(dds)")
  }
  intgroup.df <- as.data.frame(colData(object)[, intgroup, drop = FALSE])
  group <- if (length(intgroup) > 1) {
    factor(apply(intgroup.df, 1, paste, collapse = " : "))
  }
  else {
    colData(object)[[intgroup]]
  }
  d <- data.frame(PC2 = pca$x[, 2], PC3 = pca$x[, 3], group = group, 
                  intgroup.df, name = colData(rl)[,1])
  if (returnData) {
    attr(d, "percentVar") <- percentVar[2:3]
    return(d)
  }
  ggplot(data = d, aes_string(x = "PC2", y = "PC3", color = "group", label = "name")) + geom_point(size = 3) + xlab(paste0("PC2: ", round(percentVar[2] * 100), "% variance")) + ylab(paste0("PC3: ", round(percentVar[3] * 100), "% variance")) + coord_fixed() 
  
}

pdf("PCA_After_3.pdf")
plotPCA.san(rl, intgroup = 'sample', ntop = nrow(counts(cds)))
dev.off()


In [ ]:

##To plot PC1 versus 3
plotPCA.san <- function (object, intgroup = "condition", ntop = 500, returnData = FALSE) 
{
  rv <- rowVars(assay(object))
  select <- order(rv, decreasing = TRUE)[seq_len(min(ntop, 
                                                     length(rv)))]
  pca <- prcomp(t(assay(object)[select, ]))
  percentVar <- pca$sdev^2/sum(pca$sdev^2)
  if (!all(intgroup %in% names(colData(object)))) {
    stop("the argument 'intgroup' should specify columns of colData(dds)")
  }
  intgroup.df <- as.data.frame(colData(object)[, intgroup, drop = FALSE])
  group <- if (length(intgroup) > 1) {
    factor(apply(intgroup.df, 1, paste, collapse = " : "))
  }
  else {
    colData(object)[[intgroup]]
  }
  d <- data.frame(PC1 = pca$x[, 1], PC3 = pca$x[, 3], group = group, 
                  intgroup.df, name = colData(rl)[,1])
  if (returnData) {
    attr(d, "percentVar") <- percentVar[1:3]
    return(d)
  }
  ggplot(data = d, aes_string(x = "PC1", y = "PC3", color = "group", label = "name")) + geom_point(size = 3) + xlab(paste0("PC1: ", round(percentVar[1] * 100), "% variance")) + ylab(paste0("PC3: ", round(percentVar[3] * 100), "% variance")) + coord_fixed() 
  
}

pdf("PCA_After_4.pdf")
plotPCA.san(rl, intgroup = 'sample', ntop = nrow(counts(cds)))
dev.off()


In [ ]:

####

pdf("InitialHeatMap.pdf")
heatmap.2(cor(mircounts),trace="none",col=hmcol,main="Sample Correlation")
dev.off()


In [ ]:

#################

##Trying to get the Loadings please change rl.sub1 as appropriate!
##((This make the matrix for the loadings - so what I can do is fish out the miRNA of interest and plot just these miRNA names on a ggplotXY... try to do this.))
rv <- rowVars(assay(rl.sub1))
select <- order(rv, decreasing = TRUE)[seq_len(min(nrow(counts(cds)), 
                                                   length(rv)))]
pca <- prcomp(t(assay(rl.sub1)[select, ]))
loadings_Beginning_1 <- as.data.frame(pca$rotation)
write.csv(loadings_Beginning_1, file="loadings_Beginning_1_Overall.csv")

#3Then select interesting ones for PC1 and 2 via EXCEL, and then bring back here giving name columna a name already:
loadings_PC1_PC2 <- read.table("loadings_Beginning_1.txt",header=TRUE)
colnames(loadings_PC1_PC2)

library(ggrepel)
ggplot(loadings_PC1_PC2,aes(PC1,PC2,label=name))+
  theme_bw()+
  geom_text_repel(size = 3,max.overlaps = Inf)


In [ ]:

##And the Spearman Dendogram
foo_sub_1 = counts(cds[ , cds$sample %in% c("4i_hESC", "hPGCLC", "TCAM2",
                                            "hPGC_F_Wk6", "hPGC_F_Wk7", "hPGC_F_Wk9",
                                            "hPGC_M_Wk6", "hPGC_M_Wk7", "hPGC_M_Wk9",
                                            "Soma_F_Wk6", "Soma_F_Wk7", "Soma_F_Wk9",
                                            "Soma_M_Wk6", "Soma_M_Wk7", "Soma_M_Wk9")], normalized = TRUE)

foo_cor_1 = cor(foo_sub_1, method = "spearman") #builds matrix of Spearman corrn and euclidean distances

foo_cor_dist_1 = dist(foo_cor_1, method = "euclidean")

plot(hclust(foo_cor_dist_1))

In [ ]:

##NEXT IS THE TOP 15% VARIANT miRNA plot on the means. Then the subset more useful miRNA plot
library(pheatmap)

foo_sub_1 <- as.data.frame.matrix(foo_sub_1) #convert from matrix to data frame
## use for eg foo$`hPGC_M_Wk9_n1` <- foo$`hPGC-M-Wk9_2` if only want one biol replicate

colnames(foo_sub_1)

foo_sub_1$`4i_hESC_n2` <- apply(foo_sub_1[,1:2], 1, mean)
foo_sub_1$`hPGCLC_n2` <- apply(foo_sub_1[,3:4], 1, mean)
foo_sub_1$`TCAM2_n2` <- apply(foo_sub_1[,5:6], 1, mean)
foo_sub_1$`hPGC_F_n6` <- apply(foo_sub_1[,7:12], 1, mean)
foo_sub_1$`hPGC_M_n6` <- apply(foo_sub_1[,13:18], 1, mean)
foo_sub_1$`Soma_F_n6` <- apply(foo_sub_1[,19:24], 1, mean)
foo_sub_1$`Soma_M_n6` <- apply(foo_sub_1[,25:30], 1, mean)

head(foo_sub_1)

ncol(foo_sub_1)

foo_sub1_mean <- foo_sub_1[,31:37]
colnames(foo_sub1_mean)
foo_sub1_mean$sd_perGroup <- apply(foo_sub1_mean,1,sd,1:7)
hist(log1p(foo_sub1_mean$sd_perGroup),breaks = 200)

head(foo_sub1_mean)

foo_sub1_mean$sd_perGroup <- apply(foo_sub1_mean,1,sd,1:7) ##compute variance

##decide what is top 15%

nrow(foo_sub1_mean) ##calculate how many is 15% - then find that number below by trial and error aim for 280

nrow(foo_sub1_mean[foo_sub1_mean$sd_perGroup>242,]) ##this is 15% by chance, 280 miRNAs

foo_sub1_mean_top15sd <- foo_sub1_mean[foo_sub1_mean$sd_perGroup>242,1:7]
colnames(foo_sub1_mean_top15sd)

foo_sub1_mean_top15sd$ID<-rownames(foo_sub1_mean_top15sd) ##naming the miRNA rownames into a column

In [ ]:

##For log with predeterminde column orderr use cluster_cols = F

pheatmap(log1p(foo_sub1_mean_top15sd[,1:7]), border_color = NA,show_rownames = T, cluster_cols = F, labels_row = foo_sub1_mean_top15sd$ID,cutree_rows = 1,fontsize_row = 4, color = hmcol)


In [ ]:

##for z-score
myCol = colorRampPalette(brewer.pal(9,"PRGn"))(100)

pheatmap(foo_sub1_mean_top15sd[,1:7], border_color = NA,show_rownames = T,scale = "row",labels_row = foo_sub1_mean_top15sd$ID,cutree_rows = 4,fontsize_row = 1, cluster_cols = F, color = myCol)



In [ ]:

##Get the whole Foo counts table out for later line plots on Prism
foo <- counts(cds, normalized = TRUE)
write.csv(foo, file="norm_counts.csv")


In [ ]:

##Now switch gear to get the DESEq2 done but for specific categories.

#1- all (hPGC and PGCLC) up and down versus 4i-hESC and (hPGC and hPGC) up versus Soma


head(mircounts)


sampletype <- c("primed-4i", "primed-4i", "germ_art", "germ_art","germ-cancer", "germ-cancer",
                "germ","germ","germ", "germ","germ", "germ",
                "germ", "germ","germ", "germ","germ", "germ",
                "soma", "soma","soma", "soma", "soma", "soma",
                "soma", "soma", "soma", "soma", "soma", "soma",
                "primed-e8", "primed-e8", "primed", "primed","naive","naive",
                "germ-old","germ-old", "germ-old", "germ-old", 
                "germ-old", "germ-old", "germ-old",
                "germ-old", "germ-old", "germ-old", "germ-old")


conds_new = data.frame(samplenames,sampletype)
colnames(conds_new)=c("sample","type")
conds_new
